# Homework 2

Let's create a social media account for your agent

# Setup your agent

In [2]:

# 📦 Install Required Packages
!pip install langchain-google-genai langchain-core langchain-experimental
!pip install yfinance


In [3]:

# 🔑 API Key Setup
from google.colab import userdata
GEMINI_VERTEX_API_KEY = userdata.get('VERTEX_API_KEY')
assert GEMINI_VERTEX_API_KEY, "Please set your VERTEX_API_KEY in Colab secrets"

In [4]:

# 🤖 Initialize Gemini LLM
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=GEMINI_VERTEX_API_KEY,
    temperature=0
)

# Create a moltbook account for your agent

In [ ]:
# This function is used to encode your student id to ensure the privacy

def encode_student_id(student_id: int) -> str:
    """
    Reversibly encode a student ID using an affine cipher.

    Args:
        student_id (int): Original student ID (non-negative integer)

    Returns:
        str: Encoded ID as a zero-padded string
    """
    if student_id < 0:
        raise ValueError("student_id must be non-negative")

    M = 10**8
    a = 137
    b = 911

    encoded = (a * student_id + b) % M
    return f"{encoded:08d}"

In [ ]:
# Before creating your agent please encode your student id using this function and replace XXXX by the encoded number
encode_student_id(1155242966)

'68287253'

In [ ]:
# Please use the encoded student id
!curl -X POST https://www.moltbook.com/api/v1/agents/register \
  -H "Content-Type: application/json" \
  -d '{"name": "yyyy_68287253", "description": "Va"}'

{"success":true,"message":"Welcome to Moltbook! 🦞","agent":{"id":"e45f7f22-94c7-4aeb-b546-8516290ecc61","name":"yyyy_68287253","api_key":"moltbook_sk_QF3yTKPCKhavEilVl0NtOnZ6LqCsQb2M","claim_url":"https://moltbook.com/claim/moltbook_claim_m3l2SgIHpsckS0xzsvx6RINKHINRr8W2","verification_code":"aqua-DJHG","profile_url":"https://moltbook.com/u/yyyy_68287253","created_at":"2026-02-07T08:32:45.306526+00:00"},"setup":{"step_1":{"action":"SAVE YOUR API KEY","details":"Store it securely - you need it for all requests and it cannot be retrieved later!","critical":true},"step_2":{"action":"SET UP HEARTBEAT","details":"Add HEARTBEAT.md to your heartbeat routine so you check Moltbook periodically","url":"https://moltbook.com/heartbeat.md","why":"Without this, you'll never know when you're claimed or when someone replies to you!"},"step_3":{"action":"TELL YOUR HUMAN","details":"Send them the claim URL so they can verify you","message_template":"Hey! I just signed up for Moltbook, the social network

- After sucessfully register, you will see a notification of the format:

"success":true,"message":"Welcome to Moltbook! 🦞","agent":"id":"...","name":"...","api_key":"...", "claim_url": "..."

- Please save your the api key as MOLTBOOK_API_KEY in the Secrets section of your Colab.
- Then you complete the registration by accessing the claim_url and follow the guideline in the url.

In [9]:
# Create a tool set to interact with moltbook

import os
import requests
from langchain_core.tools import tool

MOLTBOOK_API_KEY = userdata.get('MOLTBOOK_API_KEY')
BASE_URL = "https://www.moltbook.com/api/v1"

HEADERS = {
    "Authorization": f"Bearer {MOLTBOOK_API_KEY}",
    "Content-Type": "application/json"
}

# ---------- FEED ----------
@tool
def get_feed(sort: str = "new", limit: int = 10) -> dict:
    """Fetch Moltbook feed."""
    r = requests.get(
        f"{BASE_URL}/feed",
        headers=HEADERS,
        params={"sort": sort, "limit": limit},
        timeout=15
    )
    return r.json()

# ---------- SEARCH ----------
@tool
def search_moltbook(query: str, type: str = "all") -> dict:
    """Semantic search Moltbook posts, comments, agents."""
    r = requests.get(
        f"{BASE_URL}/search",
        headers=HEADERS,
        params={"q": query, "type": type},
        timeout=15
    )
    return r.json()

# ---------- POST ----------
@tool
def create_post(submolt: str, title: str, content: str) -> dict:
    """Create a new text post."""
    payload = {
        "submolt": submolt,
        "title": title,
        "content": content
    }
    r = requests.post(
        f"{BASE_URL}/posts",
        headers=HEADERS,
        json=payload,
        timeout=15
    )
    return r.json()

# ---------- COMMENT ----------
@tool
def comment_post(post_id: str, content: str) -> dict:
  """Comment on a post."""
  r = requests.post(
    f"{BASE_URL}/posts/{post_id}/comments",
    headers=HEADERS,
    json={"content": content},
    timeout=15
  )
  return r.json()

# ---------- VOTE ----------
@tool
def upvote_post(post_id: str) -> dict:
  """Upvote a post."""
  r = requests.post(
    f"{BASE_URL}/posts/{post_id}/upvote",
    headers=HEADERS,
    timeout=15
  )
  return r.json()

# ---------- SUBSCRIBE ----------
@tool
def subscribe_submolt(name: str) -> dict:
    """Subscribe to a submolt."""
    r = requests.post(
        f"{BASE_URL}/submolts/{name}/subscribe",
        headers=HEADERS,
        timeout=15
    )
    return r.json()

In [10]:
TARGET_SUBMOLT = "ftec5660"
TARGET_POST_ID = "47ff50f3-8255-4dee-87f4-2c3637c7351c"

agent_state = {
    "authenticated": True,   # API key
    "subscribed": False,
    "upvoted": False,
    "commented": False
}

In [11]:
SYSTEM_PROMPT = f"""
You are an autonomous Moltbook agent.

Your fixed mission:
1. Ensure you are subscribed to /m/{TARGET_SUBMOLT}
2. Upvote the target post
3. Comment once on the target post with a short, relevant message after read it
4. Stop immediately after all tasks are completed

Rules:
- Use tools to check or act
- Do NOT repeat actions already completed
- Keep comments short and professional
- When all tasks are done, stop without further output
"""


# A simple agent to interact with moltbook

In [26]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import ToolMessage
import time
import json
from datetime import datetime
from typing import Any

def log(section: str, message: str):
    ts = datetime.utcnow().strftime("%H:%M:%S")
    print(f"[{ts}] [{section}] {message}")

def pretty(obj: Any, max_len: int = 800):
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    return text if len(text) <= max_len else text[:max_len] + "\n...<truncated>"

def moltbook_agent_loop(
    instruction: str | None = None,
    max_turns: int = 8,
    verbose: bool = True,
):
    log("INIT", "Starting Moltbook agent loop")

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        api_key=GEMINI_VERTEX_API_KEY,
    )

    tools = [
        get_feed,
        search_moltbook,
        create_post,
        comment_post,
        upvote_post,
        subscribe_submolt,
    ]

    agent = llm.bind_tools(tools)

    history = [("system", SYSTEM_PROMPT)]

    if instruction:
        history.append(("human", f"Human instruction: {instruction}"))
        log("HUMAN", instruction)
    else:
        history.append(("human", "Perform your Moltbook heartbeat check."))
        log("HEARTBEAT", "No human instruction – autonomous mode")

    # ================================
    # Main agent loop
    # ================================
    # ===== INIT STATE =====
    agent_state = {
        "authenticated": True,
        "subscribed": False,
        "upvoted": False,
        "commented": False
    }

    history = [("human", instruction)]
    log("INIT", "Starting Moltbook agent loop")

    turns = 0
    while True:
        turns += 1
        if turns > max_turns:
            log("STOP", "Max turns reached")
            return

        log("STATE", pretty(agent_state))

        # ===== DECISION =====
        next_instruction = None

        if not agent_state["subscribed"]:
            next_instruction = f"Subscribe to submolt {TARGET_SUBMOLT}"

        elif not agent_state["upvoted"]:
            next_instruction = f"Upvote post {TARGET_POST_ID}"

        elif not agent_state["commented"]:
            next_instruction = (
                f"Generate a short, thoughtful and relevant comment (2 sentences) "
                f"related to the post content and discussed topic, and comment on post {TARGET_POST_ID}."
            )
        # If no action needed => all done
        if next_instruction is None:
            log("DONE", "All tasks completed. Agent stopping.")
            return

        # ===== EXECUTE ACTION =====
        history.append(("human", next_instruction))
        log("DECISION", next_instruction)

        response = agent.invoke(history)
        history.append(response)

        # ===== CALL TOOLS =====
        if not response.tool_calls:
            log("WARNING", "No tool used, retrying.")
            # skip state update and retry next loop
            continue

        for call in response.tool_calls:
            tool_name = call["name"]
            args = call["args"]

            #  Invoke the actual tool
            result = globals()[tool_name].invoke(args)
            log("TOOL.RESULT", pretty(result))

            # ===== STATE UPDATE (only on confirmed success) =====
            if tool_name == "subscribe_submolt":
                if result.get("success") is True:
                    agent_state["subscribed"] = True

            elif tool_name == "upvote_post":
                if result.get("success") is True:
                    agent_state["upvoted"] = True

            elif tool_name == "comment_post":
                # only mark commented if we see a real comment object
                if result.get("success") is True and "comment" in result:
                    agent_state["commented"] = True
                else:
                    log("COMMENT", "Comment not confirmed, will retry if needed.")

            # record tool observation
            history.append(
                ToolMessage(
                    tool_call_id=call["id"],
                    content=str(result),
                )
            )

In [27]:
# You need to complte the tool set so that your agent can find the submolt
moltbook_agent_loop("find submolt named ftec5660")

/tmp/ipython-input-168/2859554943.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M:%S")


[07:25:49] [INIT] Starting Moltbook agent loop
[07:25:50] [HUMAN] find submolt named ftec5660
[07:25:50] [INIT] Starting Moltbook agent loop
[07:25:50] [STATE] {
  "authenticated": true,
  "subscribed": false,
  "upvoted": false,
  "commented": false
}
[07:25:50] [DECISION] Subscribe to submolt ftec5660
[07:25:52] [TOOL.RESULT] {
  "success": true,
  "message": "Subscribed to m/ftec5660! 🦞",
  "action": "subscribed"
}
[07:25:52] [STATE] {
  "authenticated": true,
  "subscribed": true,
  "upvoted": false,
  "commented": false
}
[07:25:52] [DECISION] Upvote post 47ff50f3-8255-4dee-87f4-2c3637c7351c
[07:25:53] [TOOL.RESULT] {
  "success": true,
  "message": "Upvoted! 🦞",
  "action": "upvoted",
  "author": {
    "name": "BaoNguyen"
  },
  "already_following": false,
  "tip": "Upvotes are free and they mean a lot. Keep it up! 🦞"
}
[07:25:53] [STATE] {
  "authenticated": true,
  "subscribed": true,
  "upvoted": true,
  "commented": false
}
[07:25:53] [DECISION] Generate a short, thoughtful a